# Tiny GPT, step by step
Run each cell with **Shift+Enter**, top to bottom. Read the text above each cell first.
If something breaks, restart with *Kernel → Restart Kernel* and run from the top again.

## 1. Check the GPU
You should see `True` and `NVIDIA GeForce RTX 3050 ...`. If you see `False`, PyTorch was installed without CUDA (go back to the setup steps).

In [1]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

torch: 2.14.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB


## 2. Get training data
Tiny Shakespeare: about 1 MB of plays. The model will learn to write text that looks like it.

In [2]:
import os, urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt", encoding="utf-8").read()
print("characters:", len(text))
print(text[:300])

characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## 3. Tokenizer: text → numbers
Neural nets only understand numbers. Here, every unique **character** gets an ID.
Real LLMs use sub-word tokens (BPE) instead, but the idea is the same.

In [3]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}   # string -> int
itos = {i: c for c, i in stoi.items()}       # int -> string
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("vocab size:", vocab_size)
print(repr("".join(chars)))
print(encode("hello"), "->", decode(encode("hello")))

vocab size: 65
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
[46, 43, 50, 50, 53] -> hello


## 4. Turn the whole text into one long tensor, and split it into train and validation sets
We train on 90% of the text. The other 10% is **validation**: text the model never trains on, which shows whether it's really learning or just memorising.

In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(data.shape, data[:20])

torch.Size([1115394]) tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])


## 5. Hyperparameters (the knobs)
These are sized for a **4 GB RTX 3050**. It's roughly a 10M-parameter model.
- `block_size`: how many characters the model can look back on (its context window)
- `n_embd`: size of each token's vector
- `n_head`: number of attention heads
- `n_layer`: number of transformer blocks stacked

Out of memory? Halve `batch_size` first.

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
block_size = 256
batch_size = 32
n_embd     = 384
n_head     = 6
n_layer    = 6
dropout    = 0.2
lr         = 3e-4
max_iters  = 5000
eval_every = 250
torch.manual_seed(1337)

## 6. Batches: what the model actually trains on
We take random chunks of text. **x** is the input and **y** is the same chunk shifted by one character.
The whole task of a GPT is: *given x, predict the next character at every position.*

In [6]:
def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x:", repr(decode(xb[0, :40].tolist())))
print("y:", repr(decode(yb[0, :40].tolist())))   # same text, shifted by 1

x: " royalties and rights of banish'd Herefo"
y: "royalties and rights of banish'd Herefor"


## 7. The model
Three parts:
1. **Self-attention**: each character looks at the earlier characters and decides which ones matter. *Causal* means it can't peek at the future.
2. **MLP**: a small feed-forward network that processes what attention gathered.
3. **Block** = attention + MLP, with LayerNorm and residual connections (`x = x + ...`) so deep stacks still train.

The GPT = embeddings → N blocks → a final layer that outputs a score for every possible next character.

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv  = nn.Linear(n_embd, 3 * n_embd)   # query, key, value in one go
        self.proj = nn.Linear(n_embd, n_embd)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(n_embd, dim=2)
        q, k, v = (t.view(B, T, n_head, C // n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
                                 nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)   # what the character is
        self.pos_emb = nn.Embedding(block_size, n_embd)   # where it is
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.head    = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        logits = self.head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, min(top_k, vocab_size))
            logits[logits < v[:, [-1]]] = -float("inf")
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
        return idx

model = GPT().to(device)
print("parameters: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

parameters: 10.77M


In [ ]:
model.load_state_dict(torch.load("tiny_gpt.pt", map_location=device))
model.eval()
print("trained model loaded")

## 8. Before training: pure gibberish
The weights are random, so the output is random characters. Remember what this looks like.

In [11]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, 200)[0].tolist()))


YJf,:Eik$bR&KspNx-:Em pMhNbwsVdJVm!J:jJkXjh
mMZg$MYDDc:V'ZZ.V,ZmF:eDJoVMD Kf&vDz wJ.
SIVvRpj,NibKDM$mZPi&bWmxmv?VI,CKkwm&FHLJQZ?k$zN.MV?YFImZ!bVJJ:UNxX&b:jDjPhUFtmtDpi$$$xhjJmUl:NeBjb!:Zx.:rDCi!&.dpwZ


## 9. Train
Each step: grab a batch, predict, measure how wrong the predictions are (**loss**), nudge every weight to be a bit less wrong (**backprop + AdamW**).

Watch the numbers: the loss starts around **4.2** and should drop to roughly **1.5 or lower**. If **val** stops falling while **train** keeps falling, the model is memorising (overfitting).

`autocast` uses 16-bit floats on the GPU, which is faster and uses less VRAM. On a 3050 this should take something like 10–20 minutes. You can stop it any time with the ■ button; the model keeps what it has learned so far.

In [12]:
import time
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for s in ("train", "val"):
        ls = []
        for _ in range(50):
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                ls.append(model(*get_batch(s))[1].item())
        out[s] = sum(ls) / len(ls)
    model.train()
    return out

t0 = time.time()
for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss()
        print(f"step {it:5d} | train {l['train']:.3f} | val {l['val']:.3f} | {time.time()-t0:.0f}s")
    x, y = get_batch("train")
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()

step     0 | train 4.244 | val 4.244 | 2s
step   250 | train 2.378 | val 2.408 | 27s
step   500 | train 1.978 | val 2.080 | 52s
step   750 | train 1.684 | val 1.854 | 77s
step  1000 | train 1.535 | val 1.716 | 102s
step  1250 | train 1.450 | val 1.651 | 128s
step  1500 | train 1.375 | val 1.610 | 154s
step  1750 | train 1.331 | val 1.584 | 180s
step  2000 | train 1.291 | val 1.539 | 206s
step  2250 | train 1.266 | val 1.530 | 233s
step  2500 | train 1.242 | val 1.514 | 258s
step  2750 | train 1.207 | val 1.499 | 284s
step  3000 | train 1.185 | val 1.502 | 310s
step  3250 | train 1.161 | val 1.478 | 336s
step  3500 | train 1.136 | val 1.498 | 362s
step  3750 | train 1.116 | val 1.483 | 387s
step  4000 | train 1.105 | val 1.469 | 413s
step  4250 | train 1.086 | val 1.483 | 438s
step  4500 | train 1.065 | val 1.493 | 464s
step  4750 | train 1.045 | val 1.474 | 490s
step  5000 | train 1.027 | val 1.485 | 516s


## 10. After training: generate text

In [13]:
model.eval()
start = torch.tensor([encode("ROMEO:")], device=device)
print(decode(model.generate(start, 600, temperature=0.8)[0].tolist()))

ROMEO:
But so twenty that I have sent not to do,
To content them, the room of her complaints,
And give me peace with a brancher straight
To sweet forth thy subjects' tedious words:
Now state patience in thy chair, which else art thou may,
That reason he doth so command, and to fight the cause
For his late adopting day dissolution,
Can case him to a bear himself for her weeds,
And bear in his curvation straight gainst this villain.

JULIET:
This is the Earl of Warwick, and I first,
And fear his come to be hell'd affrighted to right.

KING RICHARD II:
And am I hang'd my honour, wife, Lady Hastings?

J


## 11. Save and load
Saves the trained weights so you don't have to retrain next time.

In [14]:
torch.save(model.state_dict(), "tiny_gpt.pt")
# to load later (run cells 1-7 first):
# model.load_state_dict(torch.load("tiny_gpt.pt")); model.eval()

## 12. Experiments (this is where you actually learn)
Change one thing, re-run from cell 5, and compare the loss and the samples:
1. `temperature` in cell 10: try 0.3 vs 1.2. What changes?
2. Set `n_layer = 2`. How much worse does it get?
3. Set `block_size = 32`. Does the text lose long-range structure?
4. Swap `input.txt` for your own text (song lyrics, code, WhatsApp exports) and see what it learns to imitate.
5. Watch VRAM in another terminal with `nvidia-smi -l 1` while training.